In [ ]:
import time
import numpy as np
import polars as pl
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler

import optuna
from sklearn.svm import LinearSVC


In [ ]:
# ==========================================
# 2. PREPARACION DATASET
# ==========================================

TARGET_COL = "label"

df_encoded = pl.read_csv("../../DATASETS/dataSets_Reducidos/iot-23/datos_IOT_23_preparado.csv")

# Separación de características (X) y variable objetivo (y)
feature_columns = [col for col in df_encoded.columns if col != TARGET_COL]
X = df_encoded.select(feature_columns)
y_np = df_encoded[TARGET_COL].to_numpy().astype(np.int8)
X_np = X.to_numpy()

display(X.head())

indices = np.arange(X_np.shape[0])

train_full_idx, test_idx = train_test_split(
    indices,
    test_size=0.2,
    random_state=42,
    stratify=y_np,
)

train_idx, val_idx = train_test_split(
    train_full_idx,
    test_size=0.2,
    random_state=42,
    stratify=y_np[train_full_idx],
)

X_full_train_np = X_np[train_full_idx]
X_test_np = X_np[test_idx]
y_full_train = y_np[train_full_idx]
y_test_np = y_np[test_idx]

X_train_np = X_np[train_idx]
X_val_np = X_np[val_idx]
y_train_np = y_np[train_idx]
y_val_np = y_np[val_idx]

print(f"Entrenamiento: {len(X_train_np):,} muestras")
print(f"Validación:    {len(X_val_np):,} muestras")
print(f"Test:          {len(X_test_np):,} muestras")
print(f"Clases en test: {np.unique(y_test_np)}")
print(f"Total muestras: {len(X_np):,}")

In [ ]:
# ==========================================
# BLOQUE 1: PREPARACIÓN GLOBAL Y ESCALADO (VITAL PARA SVM)
# ==========================================
# 1. Forzamos la conversión a matriz plana
X_train_np = np.array(X_full_train_np)
y_train_np = np.array(y_full_train)

# 2. Etiquetas 0/1
y_train_01 = ((y_train_np + 1) // 2).astype(np.int8)

# 3. El escalado se hará dentro de cada fold para evitar data leakage
print("Preparando el SVM con escalado por fold...")

# ==========================================
# BLOQUE 2: FUNCIÓN OBJECTIVE
# ==========================================
def objective(trial):
    
    # 2.1 Espacio de búsqueda para LinearSVC
    # El parámetro C suele buscarse en escala logarítmica (de muy pequeño a muy grande)
    C_val = trial.suggest_float("C", 1e-4, 1e2, log=True)
    
    # 2.2 Configuración CV (3 Folds)
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    f1_scores = []
    latencies = [] 

    # 2.3 Bucle de Entrenamiento y Medición
    for train_idx, val_idx in skf.split(X_train_np, y_train_01):
        X_train_cv_raw, X_val_cv_raw = X_train_np[train_idx], X_train_np[val_idx]
        y_train_cv, y_val_cv = y_train_01[train_idx], y_train_01[val_idx]

        scaler = StandardScaler()
        X_train_cv = scaler.fit_transform(X_train_cv_raw)
        X_val_cv = scaler.transform(X_val_cv_raw)

        # dual=False es un truco de optimización brutal cuando tienes 
        # más filas (paquetes) que columnas (features)
        model = LinearSVC(
            C=C_val,
            dual=False, 
            random_state=42,
            max_iter=2000 # Le damos margen para converger
        )

        model.fit(X_train_cv, y_train_cv)

        # 1. Predecir y guardar Eficacia (F1)
        y_pred = model.predict(X_val_cv)
        f1_scores.append(f1_score(y_val_cv, y_pred, average="macro"))
        
        # 2. Medir Eficiencia (Latencia)
        subset = min(20000, len(X_val_cv))
        X_lat = X_val_cv[:subset]
        
        _ = model.predict(X_lat[:500]) # Warm-up
        
        rep_lat = []
        for _ in range(5):
            t0 = time.perf_counter()
            _ = model.predict(X_lat)
            t1 = time.perf_counter()
            rep_lat.append((t1 - t0) / len(X_lat) * 1000)
            
        latencies.append(float(np.mean(rep_lat)))

    avg_f1 = float(np.mean(f1_scores))
    avg_lat = float(np.mean(latencies))
    trial.set_user_attr("f1_std", float(np.std(f1_scores)))

    return avg_f1, avg_lat

# ==========================================
# BLOQUE 3: EJECUCIÓN DEL ESTUDIO
# ==========================================
study = optuna.create_study(directions=["maximize", "minimize"], study_name="linearsvc_ids_optimization")
print("🚀 Iniciando barrido multiobjetivo con LinearSVC (CPU)...")

study.optimize(objective, n_trials=50) 

# ==========================================
# BLOQUE 4: EXTRACCIÓN Y PARETO
# ==========================================
pareto_ids = {t.number for t in study.best_trials}
trials_data = []
for t in study.trials:
    if t.state == optuna.trial.TrialState.COMPLETE:
        trials_data.append({
            "C": t.params["C"],
            "f1_macro": t.values[0],
            "f1_std": t.user_attrs["f1_std"],
            "latency_ms": t.values[1],
            "is_pareto": t.number in pareto_ids
        })

df_results = pl.DataFrame(trials_data)
df_results.write_csv("linearsvc_trials_results_cv.csv")

print("\n✅ Resultados guardados en 'linearsvc_trials_results_cv.csv'")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

df_svm = pl.read_csv("linearsvc_trials_results_cv.csv")

plt.figure(figsize=(12, 7))
sns.set_style("whitegrid")

sns.scatterplot(
    x=df_svm["latency_ms"].to_numpy(),
    y=df_svm["f1_macro"].to_numpy(),
    hue=df_svm["is_pareto"].to_numpy(),
    palette={True: "red", False: "royalblue"},
    style=df_svm["is_pareto"].to_numpy(),
    markers={True: "D", False: "o"},
    s=100,
    alpha=0.8
)

pareto_points = df_svm.filter(pl.col("is_pareto") == True)

for row in pareto_points.iter_rows(named=True):
    plt.text(
        row["latency_ms"],
        row["f1_macro"] + 0.0005,
        f"C={row['C']:.4f}",
        fontsize=9, fontweight="bold", ha="center"
    )

plt.title("Análisis Multiobjetivo (LinearSVC): Eficacia (F1) vs. Latencia", fontsize=15)
plt.xlabel("Latencia (ms/muestra)", fontsize=12)
plt.ylabel("F1-Score Macro", fontsize=12)
plt.legend(title="¿Es Pareto?")

plt.show()

In [ ]:
# filtramos únicamente los ensayos marcados como Pareto
pareto_df = df_svm.filter(pl.col("is_pareto") == True)

plt.figure(figsize=(8, 6))
sns.set_style("whitegrid")

# un único scatter con los puntos de Pareto
sns.scatterplot(
    x=pareto_df["latency_ms"].to_numpy(),
    y=pareto_df["f1_macro"].to_numpy(),
    s=120,
    marker="D",
    color="crimson",
)

# anotamos el valor de C sobre cada marca
for row in pareto_df.iter_rows(named=True):
    plt.text(
        row["latency_ms"],
        row["f1_macro"] + 0.0005,
        f"C={row['C']:.2e}",
        fontsize=9,
        ha="center",
        fontweight="bold"
    )

plt.title("Sólo puntos de Pareto – LinearSVC", fontsize=14)
plt.xlabel("Latencia (ms/muestra)", fontsize=12)
plt.ylabel("F1‑Score Macro", fontsize=12)
plt.show()

# ahora que ya hemos generado el gráfico, veamos en bruto qué ensayos
# componen la frontera de Pareto y cuáles son sus valores

print("Puntos en la frontera de Pareto:")
display(pareto_df)

In [ ]:
import time
import numpy as np
import polars as pl
from sklearn.metrics import f1_score, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

# Convertimos las etiquetas de -1/1 a 0/1 para que las métricas funcionen igual
y_train_01 = ((y_train_np + 1) // 2).astype(np.int8)
y_test_np01 = ((y_test_np + 1) // 2).astype(np.int8)

# Aseguramos que el Test sea un array de NumPy
X_test_np_arr = np.array(X_test_np)

# ==========================================
# EVALUACIÓN FINAL EN TEST (3 CANDIDATOS SVM LINEAL)
# ==========================================

candidatos_svm = [
    {"C": 0.000299, "nombre": "Candidato 1"},
    {"C": 0.000458, "nombre": "Candidato 2"},
    {"C": 0.000187, "nombre": "Candidato 3"},
    {"C": 0.002037, "nombre": "Candidato 4"},
    {"C": 0.002889, "nombre": "Candidato 5"},
]

resultados_svm_test = []

print("--- EVALUACIÓN FINAL SOBRE EL SET DE TEST (Linear SVM) ---\n")

# 1. ESCALADO DE DATOS (El paso crítico)
print("Escalando datos (fit en Train, transform en Test)...")
scaler = StandardScaler()
# El scaler aprende las medias del Train y lo transforma
X_train_scaled = scaler.fit_transform(X_train_np)
# Al Test SOLO se le aplica la transformación matemática, sin aprender de él
X_test_scaled = scaler.transform(X_test_np_arr) 

# 2. BUCLE DE EVALUACIÓN
for c in candidatos_svm:
    print(f"Probando: {c['nombre']} (C={c['C']:.6f})...")
    
    # Inicializamos modelo
    model = LinearSVC(
        C=c["C"],
        dual=False,
        random_state=42,
        max_iter=2000
    )
    
    # Entrenamos con el 100% del Train escalado
    model.fit(X_train_scaled, y_train_01)
    
    # Warm-up para despertar la caché de la CPU
    _ = model.predict(X_test_scaled[:min(1000, len(X_test_scaled))])
    
    # Medición real de latencia en el Set de Test
    t0 = time.perf_counter()
    y_pred01 = model.predict(X_test_scaled)
    t1 = time.perf_counter()
    
    tiempo_total = t1 - t0
    latencia = (tiempo_total / len(y_test_np01)) * 1000
    
    f1_test = f1_score(y_test_np01, y_pred01, average="macro")
    acc_test = accuracy_score(y_test_np01, y_pred01)
    
    resultados_svm_test.append({
        "Perfil": c["nombre"],
        "C": c["C"],
        "F1_Test": float(f1_test),
        "Accuracy_Test": float(acc_test),
        "Latencia_ms": float(latencia)
    })

# 3. MOSTRAR TABLA
df_final_svm = pl.DataFrame(resultados_svm_test)
print("\n" + "="*70)
print("              TABLA COMPARATIVA FINAL TEST (Linear SVM)")
print("="*70)
print(df_final_svm)